In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from pygei.formatar import formatar_arquivo

In [2]:
pasta_script = Path.cwd()

arquivo = pasta_script.parent / "dados" / "EMPILHADO_MATRICULAS.parquet"

In [3]:
df = pd.read_parquet(arquivo)

In [4]:
df['num_ano_letivo'].unique()

<ArrowStringArray>
[               '2026',             '2026/1S',        '2026 - MEPES',
        '2026 - IASES',        '2026/1S - UP',     '2026/1S - IASES',
 '2026/1S CEEJA/NEEJA']
Length: 7, dtype: str

In [5]:
num_ano_letivo = [col for col in df.num_ano_letivo.unique() if '2026' in col]
df = df[df.num_ano_letivo.isin(num_ano_letivo)]
df = df[df['num_ano_letivo'] != '2026 - MEPES']

In [6]:
df['num_ano_letivo'].unique()

<ArrowStringArray>
[               '2026',             '2026/1S',        '2026 - IASES',
        '2026/1S - UP',     '2026/1S - IASES', '2026/1S CEEJA/NEEJA']
Length: 6, dtype: str

In [7]:
df = (
    df[  
        df.situacao_enturmacao.isin(["Em curso"]) &
        df.data_saida.isna() &
        df.situacao_matricula.isin(["Em curso"])  &
        df.data_encerramento_matricula.isna()
    ]#.drop_duplicates(subset='id_aluno', keep='first')    
)

In [8]:
def adicionar_alerta_mudanca_raca(df_empilhado):
    """
    Identifica alunos que tiveram dc_cor_raca alterada entre datas de referência.
    
    A coluna Alerta_Mudanca contém:
    "DESCRIÇÃO RAÇA/COR ALTERADA + (DATA_ANTERIOR + VALOR_ANTERIOR) + (DATA_NOVA + VALOR_NOVO)"
    """
    
    df = df_empilhado.copy()
    df['data_referencia'] = pd.to_datetime(df['data_referencia'])
    df = df.sort_values(['id_aluno', 'data_referencia'])
    
    # Normalizar valores de raça/cor
    df['dc_cor_raca_clean'] = df['dc_cor_raca'].astype(str).str.strip().str.upper()
    
    # Valores válidos de raça/cor
    valores_validos = ['INDÍGENA', 'BRANCA', 'PARDA', 'PRETA', 'AMARELA']
    
    # Detectar IDs com mudança E GUARDAR VALORES + DATAS
    mudanca_dict = {}
    
    for id_aluno, grupo in df.groupby('id_aluno'):
        grupo = grupo.sort_values('data_referencia')
        valores = grupo['dc_cor_raca_clean'].values
        datas = grupo['data_referencia'].values
        
        # Filtrar apenas valores válidos preenchidos COM suas datas
        valores_indices = [(i, v) for i, v in enumerate(valores) if v in valores_validos]
        
        if len(valores_indices) >= 2:
            indices_validos = [i for i, v in valores_indices]
            valores_validos_aluno = [v for i, v in valores_indices]
            
            # Se tem valores diferentes, houve mudança
            if len(set(valores_validos_aluno)) > 1:
                idx_anterior = indices_validos[0]
                idx_nova = indices_validos[-1]
                
                valor_anterior = valores[idx_anterior]
                valor_novo = valores[idx_nova]
                data_anterior = datas[idx_anterior]
                data_nova = datas[idx_nova]
                
                
                data_anterior_str = pd.Timestamp(data_anterior).strftime('%d/%m/%Y')
                data_nova_str = pd.Timestamp(data_nova).strftime('%d/%m/%Y')
                mudanca_dict[id_aluno] = {
                    'anterior': valor_anterior,
                    'nova': valor_novo,
                    'data_anterior': data_anterior_str,  
                    'data_nova': data_nova_str 
                }
    
    # Criar coluna com mensagem formatada
    def criar_mensagem_alerta(id_aluno):
        if id_aluno not in mudanca_dict:
            return None
        
        dados = mudanca_dict[id_aluno]
        data_ant = pd.Timestamp(dados['data_anterior']).strftime('%d/%m/%Y') if pd.notna(dados['data_anterior']) else 'N/A'
        data_nova = pd.Timestamp(dados['data_nova']).strftime('%d/%m/%Y') if pd.notna(dados['data_nova']) else 'N/A'
        valor_ant = dados['anterior'] if pd.notna(dados['anterior']) else 'N/A'
        valor_novo = dados['nova'] if pd.notna(dados['nova']) else 'N/A'
        
        return f"DESCRIÇÃO RAÇA/COR ALTERADA + ({data_ant} + {valor_ant}) + ({data_nova} + {valor_novo})"
    
    # Adicionar colunas
    df_empilhado['Alerta_Mudanca'] = df_empilhado['id_aluno'].map(criar_mensagem_alerta)
    
    df_empilhado['data_raca_anterior'] = df_empilhado['id_aluno'].map(
        lambda x: mudanca_dict.get(x, {}).get('data_anterior', np.nan)
    )
    
    df_empilhado['data_raca_nova'] = df_empilhado['id_aluno'].map(
        lambda x: mudanca_dict.get(x, {}).get('data_nova', np.nan)
    )
    
    df_empilhado['dc_cor_raca_anterior'] = df_empilhado['id_aluno'].map(
        lambda x: mudanca_dict.get(x, {}).get('anterior', np.nan)
    )
    
    df_empilhado['dc_cor_raca_nova'] = df_empilhado['id_aluno'].map(
        lambda x: mudanca_dict.get(x, {}).get('nova', np.nan)
    )
    
    # FILTRAR: manter apenas linhas com alerta
    df_empilhado = df_empilhado[df_empilhado['Alerta_Mudanca'].notna()]
    
    qtd = len(mudanca_dict)
    registros_filtrados = len(df_empilhado)
    print(f"✓ Detectados {qtd:,} alunos com mudança de raça/cor")
    print(f"✓ Filtrado para {registros_filtrados:,} registros com alerta")
    
    return df_empilhado

In [ ]:
# ============================================================================
# CHAMANDO AS FUNÇÕES (CÉLULA 12 - EXECUTAR AQUI!)
# ============================================================================

print("Carregando base empilhada para análise histórica...")
df_empilhado = pd.read_parquet(arquivo)

print(f"[1/1] Executando análise de mudanças de raça/cor...")
df_empilhado = adicionar_alerta_mudanca_raca(df_empilhado)

print("\n✓ Análises concluídas!")

Carregando base empilhada para análise histórica...
[1/1] Executando análise de mudanças de raça/cor...
✓ Detectados 1,844 alunos com mudança de raça/cor
✓ Filtrado para 174,530 registros com alerta

✓ Análises concluídas!


In [ ]:
import plotly.graph_objects as go
import os

df = df_empilhado

# =========================
# LIMPA NOMES DAS COLUNAS
# =========================

df = df.rename(columns={
    'dc_cor_raca_anterior': 'cor_anterior',
    'dc_cor_raca_nova': 'cor_atual'
})
# =========================
# PADRONIZA TEXTO
# =========================

# =========================
# REMOVE QUEM NÃO MUDOU
# =========================

df = df[
    df['cor_anterior']
    !=
    df['cor_atual']
]

# =========================
# AGRUPA TRANSIÇÕES
# =========================

fluxo = (
    df.groupby(
        ['cor_anterior', 'cor_atual']
    )
    .size()
    .reset_index(name='quantidade')
)

# =========================
# CRIA ORIGEM E DESTINO
# =========================

fluxo['origem'] = (
    fluxo['cor_anterior']
    + ' (ANTES)'
)

fluxo['destino'] = (
    fluxo['cor_atual']
    + ' (ATUAL)'
)

# =========================
# ORDEM FIXA
# =========================

labels_originais = [
    'AMARELA (ANTES)',
    'BRANCA (ANTES)',
    'PARDA (ANTES)',
    'PRETA (ANTES)',
    'INDÍGENA (ANTES)',

    'AMARELA (ATUAL)',
    'BRANCA (ATUAL)',
    'PARDA (ATUAL)',
    'PRETA (ATUAL)',
    'INDÍGENA (ATUAL)',
]

# =========================
# TOTAIS
# =========================

totais_origem = (
    fluxo.groupby('origem')['quantidade']
    .sum()
    .to_dict()
)

totais_destino = (
    fluxo.groupby('destino')['quantidade']
    .sum()
    .to_dict()
)

# =========================
# LABELS COM TOTAL
# =========================

labels = []

for label in labels_originais:

    if label in totais_origem:
        total = totais_origem[label]

    elif label in totais_destino:
        total = totais_destino[label]

    else:
        total = 0

    labels.append(
        f'{label}\n{total:,.0f}'.replace(',', '.')
    )

# =========================
# DICIONÁRIO DE ÍNDICES
# =========================

label_dict = {
    label: idx
    for idx, label in enumerate(labels_originais)
}

# =========================
# SOURCE / TARGET
# =========================

source = fluxo['origem'].map(label_dict)
target = fluxo['destino'].map(label_dict)

# =========================
# CORES DOS NÓS
# =========================

node_colors = [
    '#FFD700',  # AMARELA
    '#D3D3D3',  # BRANCA
    '#A0522D',  # PARDA
    '#000000',  # PRETA
    '#228B22',  # INDÍGENA

    '#FFD700',
    '#D3D3D3',
    '#A0522D',
    '#000000',
    '#228B22',
]

# =========================
# CORES DOS LINKS
# =========================

cor_links = fluxo['cor_anterior'].map({
    'AMARELA': 'rgba(255,215,0,0.5)',
    'BRANCA': 'rgba(211,211,211,0.5)',
    'PARDA': 'rgba(160,82,45,0.5)',
    'PRETA': 'rgba(0,0,0,0.5)',
    'INDÍGENA': 'rgba(34,139,34,0.5)'
})

# =========================
# GRÁFICO
# =========================

fig = go.Figure(go.Sankey(

    arrangement='snap',

    node=dict(
        pad=30,
        thickness=25,
        line=dict(color='black', width=1),
        label=labels,
        color=node_colors
    ),

    link=dict(
        source=source,
        target=target,
        value=fluxo['quantidade'],
        color=cor_links
    )
))

# =========================
# LAYOUT
# =========================

fig.update_layout(
    height=900
)

# =========================
# EXPORTA HTML
# =========================

arquivo_saida = pasta_script / "sankey_visual_v2.html"

fig.write_html(arquivo_saida)

print("Gráfico gerado com sucesso!")

Gráfico gerado com sucesso!
